# Cross-condition heatmap of significant genes

This notebook assembles gene-level log2 fold changes across six CRISPRi screen comparisons and visualizes the combined patterns as a clustered heatmap. 

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn2
from matplotlib.colors import LinearSegmentedColormap, Normalize

## 1. Define the significant-gene selection

For each comparison, the analysis retains PA locus tags represented by at least three sgRNAs, removes duplicate gene rows, and selects genes previously labelled significant by the screen-analysis workflow. Both enriched and depleted genes are included.

In [134]:
def list_siginificant(filepath):

    df = pd.read_csv(filepath, low_memory=False)

    # Use only genes that have at least 3 sgRNA
    df = df[df['nb_of_sgrna'] >= 3]
  
    # Take only PA genes (start with PA)
    df = df[(df['Gene'].str.startswith('PA'))]
    
    # Keep only one row per gene 
    df = df.drop_duplicates(subset='Gene')

    # Drop na
    df = df.dropna(subset='significant')

    # Get the enriched and depleted lists 
    df_significant = df[df['significant'] == True]

    # Keep only Gene, name, Product Name, and LFC_summary columns
    df_significant_list = set(df_significant['Gene'])
    
    return df_significant_list


## 2. Collect significant genes across conditions

Six comparisons are included: LB, glucose, and succinate at 24 and 48 hours, each relative to its corresponding LBni control. Taking the union ensures that any gene significant in at least one comparison is retained for the cross-condition visualization.

In [ ]:
# Create the lists of significant genes for all conditions

folder = Path("path/to/results/date folder")

conditions = ["24hLBni vs 24hLB.csv","24hLBni vs 24hGlc.csv", "24hLBni vs 24hSucc.csv", "48hLBni vs 48hLB.csv","48hLBni vs 48hGlc.csv", "48hLBni vs 48hSucc.csv"]

significant_lists = []
for condition in conditions: 
    filepath = folder / condition
    li = list_siginificant(filepath)
    significant_lists.append(li)



In [ ]:
# Make the union of all sets in the list

significant_genes = set.union(*significant_lists)

In [ ]:
len(significant_genes)

## 3. Build the gene-by-condition matrix

The gene-level fold change from each comparison is renamed to preserve its condition and then merged by locus tag. The resulting matrix contains one row per gene and one column per experimental comparison; it is transposed for plotting so conditions appear as rows.

In [138]:
# Get the fold change for all conditions

def get_FC(filepath, gene_list):
    
    p = Path(filepath)
    filename = p.stem
    
    df = pd.read_csv(filepath, low_memory=False)
    
    # Keep only one row per gene 
    df = df.drop_duplicates(subset='Gene')

    df_significant = df[df['Gene'].isin(gene_list)]

    df_significant = df_significant[['Gene', 'LFC_summary']]
    df_significant = df_significant.rename(columns={'LFC_summary': f'LFC_{filename}'})

    return df_significant




In [139]:
# Get the LFC for all conditions
dfs = []
for condition in conditions: 
    filepath = folder / condition
    df = get_FC(filepath, significant_genes)
    dfs.append(df)

In [ ]:
merged_df = dfs[0].copy()
for df in dfs[1:]:
    merged_df = merged_df.merge(df, how='outer', on='Gene')
merged_df.shape

In [141]:
df = merged_df.set_index('Gene')

In [ ]:
df.isna().sum()

## 4. Cluster and visualize fold-change profiles

A custom blue-to-grey-to-red color scale represents negative, near-zero, and positive fold changes. Hierarchical clustering groups genes and conditions with similar profiles. Because the normalization uses the observed global minimum and maximum, color intensity is relative to this dataset.

In [ ]:
# Create a custom colormap:
# Define a gradient from blue → grey → red
colors = [
    (0.0, "blue"),
    (0.6437, "lightblue"),
    (0.7342, "lightgrey"),   # midpoint between -1 and 1
    (0.8247, "#FFA0A0"),     # light red
    (1.0, "red")
]
custom_cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

# Normalize with plateau between -1 and 1
norm = Normalize(vmin=df.min().min(), vmax=df.max().max())

# Plot

sns.clustermap(df.T, cmap=custom_cmap, norm=norm, figsize=(8,6))

This heatmap was generated as an initial exploratory overview of fold-change patterns across conditions.

![Heat Map](figures/heatmap.png)